# Suite JRN — Journals

A journal keeps a store's history as data in a store: its journal transactions, in the order relations give them, each holding the changes one transaction made, recorded downstream of the store that made them, and read, followed, inserted into and replayed. mbse-basis [CAP-199: journals as queryable stores (mbse-journals)](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-199.md) and [CAP-288: the journal storage interface](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-288), as [the journals proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/JOURNALS.md) states.

In [ ]:
import { Builders, Changes, JSON as Json, Plain } from "@mbse/schemas/Framework";
import { ValueError } from "@mbse/schemas/Framework/Errors";

import { Journals } from "../src/Framework/index.js";

import { assert } from "./support.js";

/** crm.Person, owned by the singleton book by key, with a name. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Holds: S.OfRelation, Book: S.OfObject, Person: S.OfObject })
    .spec([
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")])
        .properties({ key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr }).adjacencies([
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true)]),
    ])]))]);
}

/** A store holding the schemas `install` defines. */
function made(install: (txn: any) => any): any {
  const store: any = new Builders.Store();
  store.transaction(install);
  return store;
}

/** A journal transaction's name in its journal. */
function name_of(transaction: any): string {
  const [entry] = [...transaction.journal];
  return entry.name.get();
}

/** The names of a journal's transactions, in order, from its first or after `after`. */
function names(journal: any, after: any = null): string[] {
  return [...Journals.transactions(journal, after)].map(name_of);
}

/** The book's contacts, each its key and name, by key. */
function contacts(store: any): [string, string | null][] {
  return [...store.singleton("crm.TheBook").contacts].map((e: any): [string, string | null] =>
    [e.key.get(), e.contact.name.get() ?? null]).sort((a, b) => a[0] < b[0] ? -1 : 1);
}

/** A transaction of `store` adding a person to its book by each key, named by it. */
function add(store: any, ...keys: string[]): void {
  const book = store.singleton("crm.TheBook");
  store.transaction(() => {
    for (const key of keys) {
      store.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key(key).book(book))).name(key));
    }
  });
}

const model = made(crm);
const journals = made(Journals.install);

<a id="JRN-01"></a>
## JRN-01 · The install defines the journals' schemas, each valid, and run again writes nothing; a journal store holds the journal's schemas alone

In [ ]:
const NAMES = ["Journals.Owns", "Journals.Directory", "Journals.Contains", "Journals.First", "Journals.Last",
  "Journals.Before", "Journals.Precedes", "Journals.Binds", "Journals.Has", "Journals.Holder", "Journals.Links",
  "Journals.OfJournal", "Journals.OfTransaction", "Journals.OfBinding", "Journals.OfEntryChange",
  "Journals.OfPropertyChange"];
const schemas = NAMES.map((name) => journals.registered(name));
assert(schemas.every((schema: any) => schema.validate().length === 0));
journals.transaction(Journals.install);
assert(NAMES.every((name, i) => journals.registered(name) === schemas[i]));
assert(!journals.names().includes("crm.Person"));

<a id="JRN-02"></a>
## JRN-02 · A journal, made once by its name, records each transaction a store commits as a journal transaction, appended in order, holding its changes

In [ ]:
const journal = Journals.journal(journals, "crm");
assert(Journals.journal(journals, "crm") === journal);
const stop = Journals.record(model, journal);
for (const key of ["ada", "bob", "cy"]) add(model, key);
assert(Json.dumps(names(journal)) === '["t1", "t2", "t3"]');
const [first, second, third] = Journals.transactions(journal);
assert(Json.dumps(Journals.changes(first)) === '{"mutations": [{"upsert": {"exists": [{"symbols": {"s0": {"reference": {"path": {"singleton": "crm.TheBook", "next": {"target": {}}}}}, "s1": {"schema": [{"$ref": {"singleton": ".", "next": {"entry": {"i": "schemas", "o": "", "properties": {"name": "crm"}, "next": {"entry": {"i": "schemas", "o": "", "properties": {"name": "Person"}, "next": {"target": {}}}}}}}}]}}, "spec": [{"s0": [{"as_adjacency": {"name": "contacts", "mutations": [{"insert": {"entries": [{"mutations": [{"as_link": {"name": "contact", "mutations": [{"set": {"ref": {"symbol": "s1"}}}]}}, {"as_property": {"name": "key", "mutations": [{"set": {"value": "ada"}}]}}]}]}}]}}]}, {"s1": [{"as_property": {"name": "name", "mutations": [{"set": {"value": "ada"}}]}}]}]}]}}]}');

<a id="JRN-03"></a>
## JRN-03 · Journal transactions are read in order, from the first or after any one; a journal with none reads none

In [ ]:
assert(Json.dumps([names(journal, first), names(journal, third), names(Journals.journal(journals, "empty"))]) === '[["t2", "t3"], [], []]');

<a id="JRN-04"></a>
## JRN-04 · A property cleared, and an entry removed by its owner's key, are held as changes and given back as written; a value is held as its plain form, the same text in both languages

In [ ]:
const ada = [...model.singleton("crm.TheBook").contacts].find((e: any) => e.key.get() === "ada").contact;
model.transaction(() => model.crm.Person(ada, (p: any) => p.name(null)));
model.transaction(() => model.crm.Book(model.singleton("crm.TheBook"), (b: any) => b.contacts(
  (a: any) => a.remove((e: any) => e.key("bob")))));
const [cleared, removed] = [...Journals.transactions(journal)].slice(-2);
assert(Json.dumps([Json.dumps(Journals.changes(cleared)), Json.dumps(Journals.changes(removed))]) === '["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"s0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"s0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"clear\\": {}}]}}]}]}]}}]}", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"s0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}, \\"spec\\": [{\\"s0\\": [{\\"as_adjacency\\": {\\"name\\": \\"contacts\\", \\"mutations\\": [{\\"remove\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"bob\\"}}]}}]}]}}]}}]}]}]}}]}"]');
add(model, "zoë");
const accented = [...Journals.transactions(journal)].at(-1);
const [held] = [...accented.changes].filter((e: any) => e.change.as_schema().name === "Journals.OfPropertyChange")
  .map((e: any) => e.change.value.get());
assert(held === '"zoë"');
model.transaction(() => model.crm.Book(model.singleton("crm.TheBook"), (b: any) => b.contacts(
  (a: any) => a.remove((e: any) => e.key("zoë")))));

<a id="JRN-05"></a>
## JRN-05 · A journal replayed into a store holding the same schemas makes it hold what the recorded store does, each journal transaction applied in order

In [ ]:
const replica = made(crm);
Journals.replay(journal, replica);
assert(Json.dumps([contacts(replica), contacts(model)]) === '[[["ada", null], ["cy", "cy"]], [["ada", null], ["cy", "cy"]]]');

<a id="JRN-06"></a>
## JRN-06 · A follower is given each journal transaction in order, from the first or after any one, then each appended, until it ends

In [ ]:
const seen: string[] = [];
const late: string[] = [];
const end = Journals.follow(journal, (t) => seen.push(name_of(t)));
const end_late = Journals.follow(journal, (t) => late.push(name_of(t)), third);
add(model, "dee");
end();
add(model, "eve");
end_late();
stop();
add(model, "fay");
assert(Json.dumps([seen, late, names(journal)]) === '[["t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8"], ["t4", "t5", "t6", "t7", "t8", "t9"], ["t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9"]]');

<a id="JRN-07"></a>
## JRN-07 · A journal transaction inserted after any one is read in its place, after the last appended; its changes applied again change nothing

In [ ]:
const inserted = Journals.insert(journal, first, Journals.changes(first));
const at_end = Journals.insert(journal, [...Journals.transactions(journal)].at(-1), Journals.changes(second));
const again = made(crm);
Journals.replay(journal, again);
assert(Json.dumps([names(journal), name_of(inserted), name_of(at_end), contacts(again)]) === '[["t1", "t10", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9", "t11"], "t10", "t11", [["ada", null], ["bob", "bob"], ["cy", "cy"], ["dee", "dee"], ["eve", "eve"]]]');

<a id="JRN-08"></a>
## JRN-08 · Two stores following one journal, each appending from what it last saw: the second append, applied where the first has, is refused, alike in both

In [ ]:
/** Applies in `target` the changes `source` reported, carried as their plain form. */
function carried(source: any, target: any, effect: any): void {
  const plain = Json.loads(Json.dumps(Plain.ToPlain(source).Transaction(effect)));
  target.transaction(Plain.FromPlain(target).Transaction(plain));
}

const [here, there] = [made(Journals.install), made(Journals.install)];
const [from_here, from_there]: any[][] = [[], []];
Changes.observe(here, (effect) => from_here.push(effect));
Changes.observe(there, (effect) => from_there.push(effect));
const mine = Journals.journal(here, "shared");
Journals.append(mine, Journals.changes(first));
for (const effect of from_here) carried(here, there, effect); // there holds what here does
const theirs = Journals.journal(there, "shared");
Journals.append(theirs, Journals.changes(second), "there-1"); // there, after the first, as it saw it
Journals.append(mine, Journals.changes(third), "here-1"); // here, after the first too
const refused: string[] = [];
for (const [source, target, effect] of [[there, here, from_there.at(-1)], [here, there, from_here.at(-1)]]) {
  try {
    carried(source, target, effect);
  } catch (error) {
    if (!(error instanceof ValueError)) throw error;
    refused.push(error.message);
  }
}
assert(Json.dumps([refused, names(mine), names(theirs)]) === '[["an entry of \'Journals.Last\' breaks unique(transaction), where a removal from it removed nothing: entries agreeing on [\'journal\'] differ on [\'transaction\']", "an entry of \'Journals.Last\' breaks unique(transaction), where a removal from it removed nothing: entries agreeing on [\'journal\'] differ on [\'transaction\']"], ["t1", "here-1"], ["t1", "there-1"]]');